# Branch Change Report

This report is generated from the current Git checkout. Run the Python cell below after pulling, committing, or editing files to refresh it.

It summarizes recent commits, changes compared with `origin/main`, and uncommitted work. Impact descriptions are intentionally short and path-based.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path.cwd()
if not (REPO / '.git').exists():
    REPO = Path(__file__).resolve().parent if '__file__' in globals() else REPO

def git(*args):
    result = subprocess.run(['git', *args], cwd=REPO, text=True, capture_output=True)
    if result.returncode != 0:
        return result.stderr.strip()
    return result.stdout.strip()

def impact(path, status):
    normalized = path.replace('\\', '/')
    if normalized.startswith('src/OOP/output/') or normalized.endswith(('.exe', '.o', '.obj')):
        return 'generated build output'
    if normalized.startswith('src/OOP/'):
        return 'OOP gameplay structure or character behavior'
    if normalized.startswith('src/DS/'):
        return 'maze data structures or traversal behavior'
    if normalized.startswith('src/CG/'):
        return 'rendering or visual output'
    if normalized.startswith('src/MP/'):
        return 'low-level math or performance support'
    if normalized.endswith(('.md', '.ipynb')):
        return 'project documentation or progress tracking'
    if normalized in {'CMakeLists.txt', 'Makefile'}:
        return 'build configuration'
    return 'project behavior or configuration'

def print_file_changes(commit_hash, prefix='  '):
    lines = git('show', '--format=', '--name-status', commit_hash).splitlines()
    for line in lines:
        parts = line.split('\t', 1)
        if len(parts) == 2:
            status, path = parts
            print(f'{prefix}- `{status}` {path}: {impact(path, status)}')

branch = git('branch', '--show-current') or 'detached HEAD'
head = git('rev-parse', '--short', 'HEAD')
print(f'## Branch: {branch} ({head})')
print(f'Repository: {REPO}')

print('\n## Recent commits and impact')
commits = git('log', '-8', '--date=short', '--pretty=format:%H|%h|%ad|%s').splitlines()
if commits and commits != ['']:
    for commit in commits:
        full_hash, short_hash, date, message = commit.split('|', 3)
        print(f'- {date} `{short_hash}`: {message}')
        print_file_changes(full_hash)
else:
    print('- No commits found.')

def changed_files(*diff_args):
    lines = git('diff', '--name-status', *diff_args).splitlines()
    return [line.split('\t', 1) for line in lines if line.strip()]

print('\n## Committed changes against origin/main')
committed = changed_files('origin/main...HEAD')
if committed:
    for status, path in committed:
        print(f'- `{status}` {path}: {impact(path, status)}')
else:
    print('- No committed file differences from origin/main.')

print('\n## Uncommitted changes')
working = changed_files('HEAD') + changed_files('--cached')
untracked = git('ls-files', '--others', '--exclude-standard').splitlines()
working += [['??', path] for path in untracked]
if working:
    seen = set()
    for status, path in working:
        if path not in seen:
            seen.add(path)
            print(f'- `{status}` {path}: {impact(path, status)}')
else:
    print('- Working tree clean.')